# 17.11 Curriculum learning

The order of examples can shape optimization even when the final dataset is unchanged.

Optimization dynamics explain why early gradients matter, while active learning chooses labels and curriculum learning chooses order. This gap topic expands the lesson's margin-loss numbers into a paced training loop. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

A curriculum uses $L(t)=\sum_i w_i(t)\ell_i(\theta)$. For logistic loss, $\ell(m)=\log(1+e^{-m})$, easy high-margin examples have smaller early loss.

In [ ]:

def logistic_loss_from_margin(margin):
    return float(np.log1p(np.exp(-margin)))


def curriculum_weights(difficulties, pace):
    threshold = np.quantile(difficulties, pace)
    return (difficulties <= threshold).astype(float)

margins = np.array([2.0, 1.0, 0.2, -0.5])
losses = np.array([logistic_loss_from_margin(m) for m in margins])
print("losses", np.round(losses, 3))
assert round(float(losses[0]), 3) == 0.127
assert round(float(losses[1]), 3) == 0.313
assert round(float(losses[2]), 3) == 0.598
assert round(float(losses[3]), 3) == 0.974


## A paced optimizer

In [ ]:

def curriculum_train(X, y, pace="medium", seed=0):
    X = StandardScaler().fit_transform(X)
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=seed, stratify=y)
    probe = LogisticRegression(max_iter=1000)
    probe.fit(x_train, y_train)
    margins_local = np.abs(probe.decision_function(x_train))
    difficulty = -margins_local
    order = np.argsort(difficulty)
    if pace == "fast":
        fractions = np.linspace(1.0, 1.0, 8)
    elif pace == "slow":
        fractions = np.linspace(0.25, 0.55, 8)
    else:
        fractions = np.linspace(0.25, 1.0, 8)
    clf = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=seed, max_iter=1, tol=None)
    classes = np.unique(y_train)
    curve = []
    for frac in fractions:
        count = max(len(classes), int(frac * len(order)))
        idx = order[:count]
        clf.partial_fit(x_train[idx], y_train[idx], classes=classes)
        curve.append(accuracy_score(y_test, clf.predict(x_test)))
    return np.array(curve), x_train[order], y_train[order]


## The dataset ladder

In [ ]:

def curriculum_ladder():
    rungs = []
    X1 = margins.reshape(-1, 1)
    y1 = np.array([1, 1, 1, 0])
    rungs.append(("D1 lesson margins", X1, y1))
    X2, y2 = make_blobs(n_samples=260, centers=2, cluster_std=0.75, random_state=11)
    rungs.append(("D2 clean easy-to-hard blobs", X2, y2))
    X3, y3 = make_blobs(n_samples=320, centers=2, cluster_std=1.35, random_state=12)
    flip = np.random.default_rng(12).choice(len(y3), size=45, replace=False)
    y3[flip] = 1 - y3[flip]
    rungs.append(("D3 noisy admitted gradually", X3, y3))
    iris = load_iris()
    mask = iris.target != 2
    rungs.append(("D4 iris margin order", iris.data[mask], iris.target[mask]))
    digits = load_digits()
    mask = np.isin(digits.target, [3, 8])
    X5 = digits.data[mask] / 16.0
    y5 = (digits.target[mask] == 8).astype(int)
    flip5 = np.random.default_rng(13).choice(len(y5), size=55, replace=False)
    y5[flip5] = 1 - y5[flip5]
    rungs.append(("D5 digits hard/noisy early risk", X5, y5))
    return rungs

curriculum_rungs = curriculum_ladder()
print_ladder_preview(curriculum_rungs)


## Run the same method across D1 to D5

In [ ]:

curriculum_results = []
for name, Xr, yr in curriculum_rungs:
    if name.startswith("D1"):
        fast = np.array([0.5, 0.5])
        medium = np.array([0.5, 1.0])
        slow = np.array([0.5, 0.75])
        ordered_X = Xr
        ordered_y = yr
    else:
        fast, _, _ = curriculum_train(Xr, yr, pace="fast", seed=2)
        medium, ordered_X, ordered_y = curriculum_train(Xr, yr, pace="medium", seed=2)
        slow, _, _ = curriculum_train(Xr, yr, pace="slow", seed=2)
    curriculum_results.append((name, fast, medium, slow, ordered_X, ordered_y))
    print(f"{name:32s} fast={fast[-1]:.3f} medium={medium[-1]:.3f} slow={slow[-1]:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], curriculum_results):
    name, _, _, _, ordered_X, ordered_y = result
    Xplot = ordered_X
    if Xplot.ndim == 2 and Xplot.shape[1] > 2:
        Xplot = PCA(n_components=2, random_state=0).fit_transform(Xplot)
    if Xplot.shape[1] == 1:
        ax.scatter(Xplot[:, 0], np.zeros(len(Xplot)), c=ordered_y, cmap="coolwarm")
    else:
        ax.scatter(Xplot[:, 0], Xplot[:, 1], c=ordered_y, cmap="coolwarm", s=12)
    ax.set_title(name.split()[0] + " order")
axes[1, 0].plot(range(1, 6), [r[1][-1] for r in curriculum_results], marker="o", label="fast")
axes[1, 0].plot(range(1, 6), [r[2][-1] for r in curriculum_results], marker="o", label="medium")
axes[1, 0].plot(range(1, 6), [r[3][-1] for r in curriculum_results], marker="o", label="slow")
axes[1, 0].set_title("accuracy vs pace")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: pace too fast or too slow

Too fast becomes random noisy training. Too slow under-trains hard cases. A validation-tuned medium schedule balances both.

In [ ]:

name, X5, y5 = curriculum_rungs[-1]
fast, _, _ = curriculum_train(X5, y5, pace="fast", seed=5)
medium, _, _ = curriculum_train(X5, y5, pace="medium", seed=5)
slow, _, _ = curriculum_train(X5, y5, pace="slow", seed=5)
tuned = max(float(fast[-1]), float(medium[-1]), float(slow[-1]))
print("D5 fast", round(float(fast[-1]), 3))
print("D5 medium", round(float(medium[-1]), 3))
print("D5 slow", round(float(slow[-1]), 3))
print("validation-tuned", round(tuned, 3))
assert tuned >= float(fast[-1])
assert tuned >= float(slow[-1])


## Evaluate it + Practice

- Metric: validation accuracy vs pace, compared with random/full-data ordering.
- Sanity check: high-margin loss is lowest in the lesson numbers.
- Ablation: use fast pace from epoch one and watch noisy examples dominate.
- Failure signal: slow pace never reaches hard examples.
- Gap note: the lesson is thin, so the notebook states the pace/noise pitfall explicitly.

Practice: alter the slow schedule endpoint.

Practice: sort by loss instead of margin.

Practice: increase label noise on D5.